In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch

script_dir = Path.cwd()
if not (script_dir / "utils" / "sls_controlled_experiment.py").exists():
    candidate = Path("ad_detection/train_notebook/SLS").resolve()
    if candidate.exists():
        script_dir = candidate
sys.path.insert(0, str(script_dir.parent.parent / "train"))
sys.path.insert(0, str(script_dir / "utils"))

from SLS_Model.model import AD_SLS_Model
from SLS_Model.train import train
from utils.dataset import create_dataloaders
from utils.visualization import plot_training_curves
from XLSR_model.model import SSLModel

from sls_controlled_experiment import (
    bootstrap_binary_metrics_with_samples,
    create_profile_repeat_split,
    ensure_sls_features_for_split,
    evaluate_lu_with_predictions,
    flatten_bootstrap_summary,
    load_dict_rows,
    save_dict_rows,
    summarize_metric_rows,
    validate_profile_availability,
)

SLS_BATCH_SIZE = 4


In [ ]:
BASE_DATASET_NAME = "Pitt-origin"
TARGET_PROFILE = "ADReSSo-like"
N_REPEATS = 10
REPEAT_SEEDS = [2026 + i for i in range(N_REPEATS)]
TRAIN_SEEDS = [21, 42, 84, 168, 336]
BOOTSTRAP_REPEATS = 1000
LU_SAMPLE_SIZE = 74
TRAIN_RATIO = 0.8

MODEL_OUTPUT_DIR = script_dir.parent.parent / "models" / f"{TARGET_PROFILE}_sls_controlled_experiment"
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using {device}")


## Step 1: Validate Target Profile Availability


In [ ]:
profile_availability = validate_profile_availability(TARGET_PROFILE)
print(profile_availability)


## Step 2: Train Matched Pitt-origin Subsets

This cell only trains models and saves training artifacts. It does not test Lu.


In [ ]:
split_summaries = []
ssl_model = None

for repeat_idx, repeat_seed in enumerate(REPEAT_SEEDS):
    print("\n" + "=" * 90)
    print(f"Training repeat {repeat_idx + 1}/{N_REPEATS} | profile={TARGET_PROFILE} | subset_seed={repeat_seed}")
    print("=" * 90)

    train_csv, val_csv, split_summary = create_profile_repeat_split(
        profile_name=TARGET_PROFILE,
        repeat_idx=repeat_idx,
        repeat_seed=repeat_seed,
        train_ratio=TRAIN_RATIO,
    )
    sampled_rows = split_summary.pop("sampled_rows")
    split_summaries.append(split_summary)

    repeat_output_dir = MODEL_OUTPUT_DIR / f"repeat_{repeat_idx:02d}"
    save_dict_rows(repeat_output_dir / "subset_samples.csv", sampled_rows)

    print(f"Train CSV: {train_csv}")
    print(f"Val CSV: {val_csv}")
    print(f"Split summary: total={split_summary['total']}, train={split_summary['train_size']}, val={split_summary['val_size']}")

    ssl_model = ensure_sls_features_for_split(
        train_csv=train_csv,
        val_csv=val_csv,
        device=device,
        ssl_model=ssl_model,
    )

    train_loader = create_dataloaders(
        data_csv=train_csv,
        feature_type="sls",
        batch_size=SLS_BATCH_SIZE,
    )
    val_loader = create_dataloaders(
        data_csv=val_csv,
        feature_type="sls",
        batch_size=SLS_BATCH_SIZE,
    )

    seed_result_rows = []
    for train_seed in TRAIN_SEEDS:
        seed, metrics, history = train(
            seed=train_seed,
            train_loader=train_loader,
            val_loader=val_loader,
            output_dir=repeat_output_dir,
            device=device,
        )
        seed_result_rows.append({
            "repeat_idx": repeat_idx,
            "subset_seed": repeat_seed,
            "seed": seed,
            "val_acc": metrics["val_acc"],
            "val_loss": metrics["val_loss"],
            "control_acc": metrics["control_acc"],
            "dementia_acc": metrics["dementia_acc"],
            "f1_score": metrics["f1_score"],
        })
        save_dict_rows(repeat_output_dir / "seed_results.csv", seed_result_rows)

        history_rows = [
            {
                "repeat_idx": repeat_idx,
                "subset_seed": repeat_seed,
                "seed": seed,
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "train_acc": train_acc,
                "val_acc": val_acc,
            }
            for epoch, train_loss, val_loss, train_acc, val_acc in zip(
                history["epochs"],
                history["train_losses"],
                history["val_losses"],
                history["train_accs"],
                history["val_accs"],
            )
        ]
        save_dict_rows(repeat_output_dir / f"training_history_seed_{seed}.csv", history_rows)

        plot_training_curves(
            epochs=history["epochs"],
            train_loss=history["train_losses"],
            val_loss=history["val_losses"],
            train_acc=history["train_accs"],
            val_acc=history["val_accs"],
            title_prefix=f"Repeat {repeat_idx} Seed {seed}",
        )

    print(f"Saved seed results to {repeat_output_dir / 'seed_results.csv'}")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

save_dict_rows(MODEL_OUTPUT_DIR / "split_summaries.csv", split_summaries)
print(f"Saved split summaries to {MODEL_OUTPUT_DIR / 'split_summaries.csv'}")


## Step 3: Test Saved Models on Lu

This cell is independent of Step 2 memory state. After training finishes, it can be run after a kernel restart as long as the import/config/device cells above have been run.


In [ ]:
repeat_results = []
bootstrap_results = []
ssl_model = SSLModel(device, freeze_xlsr=True)

for repeat_idx, repeat_seed in enumerate(REPEAT_SEEDS):
    print("\n" + "=" * 90)
    print(f"Testing repeat {repeat_idx + 1}/{N_REPEATS} | profile={TARGET_PROFILE} | subset_seed={repeat_seed}")
    print("=" * 90)

    repeat_output_dir = MODEL_OUTPUT_DIR / f"repeat_{repeat_idx:02d}"
    seed_results_path = repeat_output_dir / "seed_results.csv"
    if not seed_results_path.exists():
        raise FileNotFoundError(f"Missing training results: {seed_results_path}")

    seed_rows = load_dict_rows(seed_results_path)
    best_seed_row = max(
        seed_rows,
        key=lambda row: (float(row["val_acc"]), -float(row["val_loss"])),
    )
    best_seed = int(best_seed_row["seed"])
    best_model_path = repeat_output_dir / f"seed_{best_seed}" / "best.pth"
    if not best_model_path.exists():
        raise FileNotFoundError(f"Missing best model checkpoint: {best_model_path}")
    print(f"Best seed: {best_seed}, best model: {best_model_path}")

    test_model = AD_SLS_Model()
    checkpoint = torch.load(best_model_path, map_location=device)
    test_model.load_state_dict(checkpoint)
    test_model = test_model.to(device)
    test_model.eval()

    lu_result = evaluate_lu_with_predictions(
        model=test_model,
        device=device,
        ssl_model=ssl_model,
        batch_size=SLS_BATCH_SIZE,
    )
    if lu_result["n_samples"] != LU_SAMPLE_SIZE:
        print(f"Warning: Lu prediction count is {lu_result['n_samples']}; bootstrap sample size is configured as {LU_SAMPLE_SIZE}.")

    bootstrap_summary, bootstrap_sample_rows = bootstrap_binary_metrics_with_samples(
        y_true=lu_result["y_true"],
        y_pred=lu_result["y_pred"],
        n_bootstrap=BOOTSTRAP_REPEATS,
        sample_size=LU_SAMPLE_SIZE,
        seed=repeat_seed,
    )
    bootstrap_sample_rows = [
        {"repeat_idx": repeat_idx, "subset_seed": repeat_seed, **row}
        for row in bootstrap_sample_rows
    ]
    save_dict_rows(repeat_output_dir / "bootstrap_samples.csv", bootstrap_sample_rows)

    repeat_row = {
        "repeat_idx": repeat_idx,
        "subset_seed": repeat_seed,
        "best_train_seed": best_seed,
        "best_val_acc": float(best_seed_row["val_acc"]),
        "best_val_loss": float(best_seed_row["val_loss"]),
        "best_val_f1": float(best_seed_row["f1_score"]),
        "lu_accuracy": lu_result["accuracy"],
        "lu_f1": lu_result["dementia_f1"],
        "lu_control_f1": lu_result["control_f1"],
        "lu_dementia_f1": lu_result["dementia_f1"],
        "lu_control_acc": lu_result["control_acc"],
        "lu_dementia_acc": lu_result["dementia_acc"],
        "lu_n_predictions": lu_result["n_samples"],
        "lu_bootstrap_sample_size": LU_SAMPLE_SIZE,
    }
    repeat_results.append(repeat_row)

    bootstrap_row = {"repeat_idx": repeat_idx, "subset_seed": repeat_seed}
    bootstrap_row.update(flatten_bootstrap_summary(bootstrap_summary))
    bootstrap_results.append(bootstrap_row)

    prediction_rows = [
        {
            "repeat_idx": repeat_idx,
            "session_id": session_id,
            "y_true": y_true,
            "y_pred": y_pred,
        }
        for session_id, y_true, y_pred in zip(
            lu_result["session_ids"], lu_result["y_true"], lu_result["y_pred"]
        )
    ]
    save_dict_rows(repeat_output_dir / "lu_predictions.csv", prediction_rows)

    print(f"Lu result: {repeat_row}")
    print(f"Bootstrap summary: {bootstrap_summary}")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

save_dict_rows(MODEL_OUTPUT_DIR / "repeat_results.csv", repeat_results)
save_dict_rows(MODEL_OUTPUT_DIR / "bootstrap_results.csv", bootstrap_results)
print(f"Saved repeat results to {MODEL_OUTPUT_DIR / 'repeat_results.csv'}")
print(f"Saved bootstrap results to {MODEL_OUTPUT_DIR / 'bootstrap_results.csv'}")


## Step 4: Summarize Saved Test Results


In [ ]:
repeat_results_path = MODEL_OUTPUT_DIR / "repeat_results.csv"
if not repeat_results_path.exists():
    raise FileNotFoundError(f"Missing test results: {repeat_results_path}")

repeat_results = load_dict_rows(repeat_results_path)
metrics_summary = summarize_metric_rows(
    repeat_results,
    [
        "lu_accuracy",
        "lu_control_f1",
        "lu_dementia_f1",
        "lu_control_acc",
        "lu_dementia_acc",
    ],
)
metrics_summary["lu_bootstrap_sample_size"] = LU_SAMPLE_SIZE

print("\n" + "=" * 90)
print("FINAL LU SUMMARY ACROSS REPEATS")
print("=" * 90)
for key, value in metrics_summary.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

save_dict_rows(MODEL_OUTPUT_DIR / "final_summary.csv", [metrics_summary])
metrics_summary
